# TDQE: 文本数据质量评估

**论文:** "TDQE: 一种面向深度学习的文本数据质量评估方法"  
罗春旭, 熊海旭, 叶雅珍, 丁滟, 宗世泽, 熊贇, 朱扬勇  
复旦大学 & 国防科技大学

本 Notebook 实现 TDQE 完整流水线:
1. **Section 2.1** — 鲁棒性 R(V(T)): Dropout 随机子网络 [公式 (1)]
2. **Section 2.2** — 准确性 M(T,S): 文本-摘要匹配
3. **Section 2.3** — 质量 Q(T) = R + M [公式 (2)]
4. **Section 3.1** — 按 Table 1 参数训练分类器
5. **Section 3.3–3.5** — 数据删除与消融实验

In [1]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import torch
from transformers import GPT2Tokenizer, AutoModelForCausalLM

from tdqe import (
    load_20ng, split_dataset,
    compute_all_scores,
    train_classifier, run_data_removal_experiment, run_ablation_experiment,
    MAX_INPUT_LENGTH, NUM_DROPOUT_PASSES,
    NUM_EPOCHS, BATCH_SIZE, LEARNING_RATE,
)

warnings.filterwarnings("ignore")
print("TDQE 包导入成功。")

TDQE 包导入成功。


## 第一步: 加载预训练语言模型

使用 distilgpt2 (8200 万参数)，Dropout = 0.1。首次运行自动下载 ~330 MB。

论文指出框架是模型无关的: "不同结构的模型并不会对本文提出的方法输出的数据质量评分排名产生明显影响" (Section 2.3)。

In [2]:
MODEL_NAME = "distilgpt2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"设备: {device.upper()}")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.config.output_hidden_states = True

print(f"模型: {model.config.model_type} | "
      f"隐藏维度: {model.config.n_embd} | 层数: {model.config.n_layer} | "
      f"Dropout: embd={model.config.embd_pdrop}, attn={model.config.attn_pdrop}")
print("模型就绪。")

设备: CPU


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

模型: gpt2 | 隐藏维度: 768 | 层数: 6 | Dropout: embd=0.1, attn=0.1
模型就绪。


## 第二步: 加载 20 Newsgroups 数据集 (Section 3.1)

20 个类别，约 19,000 篇文章。平均 266 词/篇。按 8:2 划分。

In [3]:
data_dir = "./archive"
texts, labels, categories = load_20ng(data_dir)
print(f"总文章数: {len(texts)}")

train_texts, train_labels, test_texts, test_labels = split_dataset(texts, labels)
print(f"训练集: {len(train_texts)} | 测试集: {len(test_texts)}")

print("\n各类别文章数:")
for cat in sorted(set(categories)):
    print(f"  {cat:35s} {sum(1 for c in categories if c == cat):5d}")

总文章数: 37048
训练集: 29638 | 测试集: 7410

各类别文章数:
  alt.atheism                          1591
  comp.graphics                        1897
  comp.os.ms-windows.misc              1940
  comp.sys.ibm.pc.hardware             1942
  comp.sys.mac.hardware                1886
  comp.windows.x                       1916
  misc.forsale                         1852
  rec.autos                            1948
  rec.motorcycles                      1968
  rec.sport.baseball                   1932
  rec.sport.hockey                     1960
  sci.crypt                            1974
  sci.electronics                      1930
  sci.med                              1948
  sci.space                            1958
  soc.religion.christian               1966
  talk.politics.guns                   1806
  talk.politics.mideast                1852
  talk.politics.misc                   1536
  talk.religion.misc                   1246


## 第三步: 计算 TDQE 质量分数

对训练集中每条文本 T:
- **R(V(T))**: m=3 次 Dropout 前向传播 → 嵌入向量两两欧氏距离均值 → σ(−avg_dist) ∈ [0, 0.5]
- **M(T,S)**: 生成摘要 S → T 与 S 嵌入的余弦相似度 → 缩放到 [0, 0.5]
- **Q(T)** = R + M ∈ [0, 1.0]

⚠ 这是最耗时的步骤。~30K 条训练数据在 CPU 上预计需要数小时。

In [ ]:
print(f"正在对 {len(train_texts)} 条训练样本打分 (m={NUM_DROPOUT_PASSES}, max_len={MAX_INPUT_LENGTH})...")
t0 = time.time()

scores = compute_all_scores(model, tokenizer, train_texts, m=NUM_DROPOUT_PASSES,
                            device=device, progress_interval=50)

elapsed = time.time() - t0
print(f"打分完成，耗时 {elapsed/3600:.1f} 小时")

# 保存分数
df_scores = pd.DataFrame(scores)
df_scores["category"] = [categories[i] for i in range(len(train_texts))]
df_scores.to_csv("tdqe_scores.csv", index=False)
print("分数已保存到 tdqe_scores.csv")

print(f"\n分数统计 (训练集):")
print(f"  鲁棒性 R: 均值 = {np.mean(scores['robustness']):.4f}, 标准差 = {np.std(scores['robustness']):.4f}")
print(f"  准确性 M: 均值 = {np.mean(scores['accuracy']):.4f}, 标准差 = {np.std(scores['accuracy']):.4f}")
print(f"  质量 Q:   均值 = {np.mean(scores['quality']):.4f}, 标准差 = {np.std(scores['quality']):.4f}")

正在对 29638 条训练样本打分 (m=3, max_len=512)...


## 第四步: 分类器验证实验 (Section 3)

Table 1 参数: epochs=10, batch=8, lr=0.01, Leaky-ReLU, max_len=512。

In [ ]:
# 基线分类器 (不删除任何数据)
print("=== 基线分类器 ===")
p0, r0 = train_classifier(model, tokenizer, train_texts, train_labels,
                          test_texts, test_labels, device=device)
print(f"基线: 准确率 P = {p0:.4f}, 召回率 R = {r0:.4f}")

In [ ]:
# 实验一: 删除低质量数据 (Section 3.3, Fig 2)
# 论文结果: 删除约 15% 最低分数据时 P/R 达到峰值
print("\n=== 删除低质量数据 ===")
fracs = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

low_results = run_data_removal_experiment(
    model, tokenizer, train_texts, train_labels, test_texts, test_labels,
    scores["quality"], fracs, remove_low=True, device=device)

print("\n删除比例    准确率 P    召回率 R")
for frac, (p, r) in zip(fracs, low_results):
    print(f"  {frac:.0%}        {p:.4f}      {r:.4f}")

In [ ]:
# 实验二: 删除高质量数据 (Section 3.3, Fig 3)
# 论文结果: 删除最高分数据导致 P/R 快速下降
print("\n=== 删除高质量数据 ===")
fracs_high = [0.00, 0.03, 0.06, 0.09, 0.12, 0.15, 0.18]

high_results = run_data_removal_experiment(
    model, tokenizer, train_texts, train_labels, test_texts, test_labels,
    scores["quality"], fracs_high, remove_low=False, device=device)

print("\n删除比例    准确率 P    召回率 R")
for frac, (p, r) in zip(fracs_high, high_results):
    print(f"  {frac:.0%}        {p:.4f}      {r:.4f}")

In [ ]:
# 实验三: 消融实验 (Section 3.5, Fig 7)
# 对比: 仅鲁棒性 vs. 仅准确性 vs. 组合 TDQE
print("\n=== 消融实验 ===")
r_only, a_only, combined = run_ablation_experiment(
    model, tokenizer, train_texts, train_labels, test_texts, test_labels,
    scores["robustness"], scores["accuracy"], fracs, device=device)

print("\n删除比例    仅鲁棒性    仅准确性    组合(TDQE)")
for i, frac in enumerate(fracs):
    print(f"  {frac:.0%}        {r_only[i][0]:.4f}      {a_only[i][0]:.4f}      {combined[i][0]:.4f}")

## 总结

以上实验复现论文核心发现:
- **Fig 2**: 删除约 15% 最低 Q(T) 数据后分类器 P/R 提升
- **Fig 3**: 删除最高 Q(T) 数据导致 P/R 迅速下降
- **Fig 7**: 组合 (R+M) 优于仅鲁棒性或仅准确性

分数已保存至 `tdqe_scores.csv`。